In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from loguru import logger

# Suppress debug messages
logger.remove()
_ = logger.add(sys.stderr, level="INFO")

In [ ]:
from pathlib import Path

from moyopy import (
    SpaceGroup,
    SpaceGroupType,
    MagneticSpaceGroupType,
    NonCollinearMagneticCell,
    MoyoNonCollinearMagneticDataset,
)
from moyopy.interface import MoyoAdapter
from pymatgen.core import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import numpy as np
from spinspg.spin import SpinOnlyGroupType

from spinforge.configuration import SSAGenerator
from spinforge.scif import SpinCifWriter
from spinforge.testing import load_prim_CoTa3S6

In [ ]:
outputs_dir = Path().resolve() / "outputs"
outputs_dir.mkdir(exist_ok=True)

# CoTa3S6

In [ ]:
# P 6_3 2 2 (No.182)
prim_cell = load_prim_CoTa3S6()
magnetic_site_indices = [
    i for i, number in enumerate(prim_cell.numbers) if number == 27
]  # Co sites (2d)

In [ ]:
scg = SSAGenerator(
    prim_cell=prim_cell,
    magnetic_site_indices=magnetic_site_indices,
)

In [ ]:
print(prim_cell.numbers)
print(scg.prim_dataset.wyckoffs)

## Spin space group analysis

In [ ]:
# SSG enumeration settings
spin_only_group_type = SpinOnlyGroupType.NONCOPLANAR
k_index = 4

In [ ]:
rng = np.random.default_rng(seed=0)

list_ms = []
for i, (sog, nssg, sas) in enumerate(
    scg.enumerate(spin_only_group_type=spin_only_group_type, k_index=k_index)
):
    print("=" * 80)
    print(f"SSG #{i=}")
    invariant_space_group = SpaceGroup(
        prim_rotations=nssg.invariant_rotations.tolist(),
        prim_translations=nssg.invariant_translations.tolist(),
        basis=prim_cell.basis,
    )
    invariant_space_group_type = SpaceGroupType(invariant_space_group.number)
    print(f"t-index: {nssg.t_index}")
    print(
        f"Invariant SG: {invariant_space_group_type.hm_short} (No. {invariant_space_group_type.number})"
    )

    print(f"k-index: {nssg.k_index}")
    print(f"Supercell transformation matrix: \n{nssg.sublattice.transformation}")

    print(f"Dimension of symmetry-adapted structure: {sas.dim}")

    # Sample a magnetic structure
    for k, magnetic_moments in enumerate(sas.magnetic_moments_basis):
        ms = sas.generate_with_magnetic_moments(magnetic_moments)
        list_ms.append(ms)

        writer = SpinCifWriter(sas, nssg, magnetic_moments, spin_only_group=sog)
        writer.write_file(str(outputs_dir / f"ssg_{i}_{k}.scif"))

## Oriented spin space group analysis

In [ ]:
for i, (sog, nssg, sas) in enumerate(
    scg.enumerate(spin_only_group_type=spin_only_group_type, k_index=k_index)
):
    print("=" * 80)
    print(f"SSG #{i=}")
    for j, (ms, msg) in enumerate(
        sas.generate_oriented(
            spin_only_group=sog,
            nontrivial_spin_space_group=nssg,
            preserve_spin_planochirality=True,
        )
    ):
        magnetic_cell = NonCollinearMagneticCell(
            basis=ms.lattice.matrix.tolist(),
            positions=ms.frac_coords.tolist(),
            numbers=list(ms.atomic_numbers),
            magnetic_moments=np.array(ms.site_properties["magmom"]).tolist(),
        )
        magnetic_dataset = MoyoNonCollinearMagneticDataset(magnetic_cell)
        magnetic_space_group_type = MagneticSpaceGroupType(magnetic_dataset.uni_number)

        print(
            f"    BNS: {magnetic_space_group_type.bns_number} ({msg.msg_type}), UNI: {magnetic_space_group_type.uni_number}"
        )
        writer = SpinCifWriter.from_oriented(sas, nssg, ms, msg, spin_only_group=sog)
        writer.write_file(str(outputs_dir / f"ssg_{i}_msg_{j}.scif"))